# Tensors & Autograd Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Name that rank.** Rank = number of axes; (8, 3, 28, 28) reads "batch of eight RGB images, 28 pixels tall and wide".

In [ ]:
import numpy as np

scalar = np.array(3.14)
vector = np.array([70.0, 180.0, 25.0])
matrix = np.array([[70.0, 180.0, 25.0],
                   [65.0, 170.0, 31.0]])
image = np.zeros((3, 28, 28))
batch = np.zeros((8, 3, 28, 28))

for name, t in [("scalar", scalar), ("vector", vector),
                ("matrix", matrix), ("image (C,H,W)", image),
                ("batch (B,C,H,W)", batch)]:
    print(f"{name:16s} ndim={t.ndim}  shape={t.shape}  dtype={t.dtype}")
# batch: B=8 images, C=3 colour channels, H=28 rows, W=28 columns

**2. The two-way creation dictionary.** Same names and behaviour in both libraries - PyTorch factories take each dimension as a separate argument, NumPy takes a shape tuple.

In [ ]:
import numpy as np

print(np.zeros((2, 3)).shape)     # torch: torch.zeros(2, 3)
print(np.ones(4))                 # torch: torch.ones(4)
print(np.arange(0, 10, 2))        # torch: torch.arange(0, 10, 2)
print(np.linspace(0, 1, 5))       # torch: torch.linspace(0, 1, 5)
print(np.eye(3))                  # torch: torch.eye(3)
# PyTorch takes dims separately: torch.zeros(3, 4), not (3, 4).

**3. dtypes: the deep-learning defaults.** NumPy guesses int64/float64; DL conventions are float32 parameters (speed + memory) and int64 class indices.

In [ ]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([1.0, 2.5])
labels = np.array([0, 1, 2], dtype=np.int64)
weights = np.array([1, 2, 3], dtype=np.float32)

print(a.dtype, b.dtype, labels.dtype, weights.dtype)
print("weights itemsize:", weights.itemsize, "byte")
# float32: ~2x GPU throughput and half the VRAM of float64;
# int64: class indices must be exact integers, never floats.

## Part 2 — Practice

**4. Shape surgery on an image batch.** `reshape` reinterprets the buffer, `transpose` permutes the axes - neither moves data, both are free.

In [ ]:
import numpy as np

imgs = np.random.default_rng(0).random((2, 3, 4, 5))
flat = imgs.reshape(imgs.shape[0], -1)
as_hwc = imgs.transpose(0, 2, 3, 1)      # (B, C, H, W) -> (B, H, W, C)

print("original:", imgs.shape)
print("flattened:", flat.shape)
print("NHWC    :", as_hwc.shape)

**5. Broadcasting: standardise and stretch.** Right-aligned shapes must be equal or contain a 1: (5,4)+(4,), (5,4)+(5,1) and (3,1)+(1,4) all broadcast.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
scores = rng.integers(40, 100, size=(5, 4)).astype(float)

standardised = (scores - scores.mean(axis=0)) / scores.std(axis=0)
print("column means now:", standardised.mean(axis=0).round(6))

row_bias = np.array([[0.5], [1.0], [1.5], [2.0], [2.5]])
curved = scores + row_bias               # (5,1) stretches over (5,4)
print("curved shape:", curved.shape)

grid = np.arange(3).reshape(3, 1) + np.arange(4).reshape(1, 4)
print(grid)

**6. Select patients like a pro.** Slices give views, masks and fancy indices give copies - tensor indexing inherits every one of these rules.

In [ ]:
import numpy as np

patients = np.array([[72.0, 120.0, 98.0],
                     [95.0, 150.0, 91.0],
                     [68.0, 118.0, 99.0],
                     [110.0, 160.0, 88.0]])

print(patients[:2])
print(patients[:, 1])
print(patients[patients[:, 0] > 90])
worst = patients[np.argmin(patients[:, 2])]
print("lowest SpO2 row:", worst)
print("reversed HR   :", patients[::-1, 0])

**7. What would autograd return?** Autograd automates this calculus: record ops during the forward pass, replay them in reverse, deposit derivatives in `.grad`.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
X = rng.normal(size=(30, 4))
y = rng.normal(size=(30, 1))
w = rng.normal(scale=0.1, size=(4, 1))

loss = np.mean((X @ w - y) ** 2)
grad = 2 / len(y) * X.T @ (X @ w - y)     # the chain rule, done once

print("loss:", round(loss, 6))
print("gradient:", grad.ravel().round(5))
# requires_grad=True -> start recording ops touching w
# loss.backward()    -> fill w.grad with exactly these values
# zero_grad()        -> clear stale .grad (it accumulates!)
# torch.no_grad()    -> stop recording at inference, saving memory

## Part 3 — Challenge

**8. Numerical gradient check.** Central differences reproduce autograd to ~8 decimals - gradient checking catches sign errors in hand-derived maths.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
X = rng.normal(size=(30, 4))
y = rng.normal(size=(30, 1))
w = rng.normal(scale=0.1, size=(4, 1))

def f(w):
    return np.mean((X @ w - y) ** 2)

eps = 1e-6
num_grad = np.zeros_like(w)
it = np.nditer(w, flags=["multi_index"])
while not it.finished:
    i = it.multi_index
    wp, wm = w.copy(), w.copy()
    wp[i] += eps
    wm[i] -= eps
    num_grad[i] = (f(wp) - f(wm)) / (2 * eps)
    it.iternext()

ana = 2 / len(y) * X.T @ (X @ w - y)
print("numerical:", num_grad.ravel().round(8))
print("analytic :", ana.ravel().round(8))
print("max abs difference:", np.abs(num_grad - ana).max())

**9. The forgotten zero_grad.** `.grad` ACCUMULATES by design, so skipping zero_grad() silently doubles every update - the ratio below is exactly 2.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
X = rng.normal(size=(30, 4))
y = rng.normal(size=(30, 1))
w = rng.normal(scale=0.1, size=(4, 1))

g = 2 / len(y) * X.T @ (X @ w - y)   # what .backward() deposits
accumulated = g + g                  # forgot zero_grad() in between

lr = 0.1
correct_step = lr * g
buggy_step = lr * accumulated
print("change magnitude (correct):", np.abs(correct_step).sum())
print("change magnitude (buggy)  :", np.abs(buggy_step).sum())
print("ratio:", np.abs(accumulated).sum() / np.abs(g).sum())
# optimizer.zero_grad() at the top of every iteration prevents this.